In [1]:
# -*- coding: utf-8 -*-
"""
Above-the-fold ablation: stratified 5-fold CV of CLIP-Base on WDP
=================================================================

Same setup as the full-page WDP cross-validation notebook, except every
screenshot is cropped to its top CFG.ABOVE_FOLD_HEIGHT pixels before
letterboxing, so the model only sees roughly what a visitor sees before
scrolling.

WDP (Web Design Prototypicality, Miniukovich & Figl 2024): 3,136 full-page
homepage screenshots in four categories (banks, fashion, homeware,
universities). https://doi.org/10.1016/j.dib.2023.109976

Source of the input-scope ablation (Table 7) in the paper.

Original (1440×3000)        Cropped (1440×1024)         Final (224×224)
┌──────────────────┐        ┌──────────────────┐       ┌────────────┐
│     Header       │        │     Header       │       │            │
│     Hero         │        │     Hero         │   →   │  Cropped   │
│     Content      │   →    │     Content      │       │  + Scaled  │
│     More...      │        └──────────────────┘       │            │
│     More...      │         (top 1024px only)         └────────────┘
│     Footer       │
└──────────────────┘
"""

import os
import random
import json
from dataclasses import dataclass
from datetime import datetime
import gc

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torch import amp

from scipy.stats import pearsonr, spearmanr
from sklearn.model_selection import StratifiedKFold

from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    get_cosine_schedule_with_warmup,
)

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# Setup
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_cuda = torch.cuda.is_available()
print(f"Device: {device}")

# ============================================================
# Config
# ============================================================
@dataclass
class Config:
    # Paths
    WDP_PATH: str = "/content/webdesignprototypicality"
    # Separate cache dir: file names would collide with the full-page WDP run
    CACHE_DIR: str = "/content/cache_wdp_above_fold"
    RESULTS_DIR: str = "/content/drive/MyDrive/results/vit_benchmark/wdp_above_fold_5fold"
    ABOVE_FOLD_HEIGHT: int = 1024

    # Categories
    CATEGORIES: tuple = ("banks", "fashion", "homeware", "universities")

    # Model (optimized hyperparameters)
    HF_MODEL: str = "openai/clip-vit-base-patch16"
    IMG_SIZE: int = 224
    BASE_LR: float = 3e-6
    HEAD_LR: float = 2e-4
    WEIGHT_DECAY: float = 0.01
    BATCH_SIZE: int = 16
    EPOCHS_HEAD: int = 5
    EPOCHS_PARTIAL: int = 5
    EPOCHS_FULL: int = 5
    WARMUP_RATIO: float = 0.1

    # CV
    N_FOLDS: int = 5
    SEED: int = 42

    # Eval
    USE_TTA: bool = False
    BOOTSTRAP_N: int = 2000

    def __post_init__(self):
        from google.colab import drive
        drive.mount('/content/drive')

        if not os.path.exists(self.WDP_PATH):
            os.makedirs(self.WDP_PATH, exist_ok=True)
            os.system(f'unzip -q /content/drive/MyDrive/datasets/webdesignprototypicality.zip -d {self.WDP_PATH}')

        for d in [self.CACHE_DIR, self.RESULTS_DIR]:
            os.makedirs(d, exist_ok=True)

CFG = Config()

# ============================================================
# Logging
# ============================================================
class Logger:
    def __init__(self):
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.log_file = os.path.join(CFG.RESULTS_DIR, f"wdp_5fold_{ts}.log")

    def log(self, msg):
        ts = datetime.now().strftime("%H:%M:%S")
        print(f"[{ts}] {msg}")
        with open(self.log_file, "a") as f:
            f.write(f"[{ts}] {msg}\n")

logger = Logger()

# ============================================================
# Data Loading
# ============================================================
def load_wdp_data():
    logger.log("Loading WDP dataset...")

    rows = []
    for cat in CFG.CATEGORIES:
        ratings_file = os.path.join(CFG.WDP_PATH, cat, f"ratings.avg.{cat}.txt")
        screenshot_dir = os.path.join(CFG.WDP_PATH, cat, "screenshots")

        if not os.path.exists(ratings_file):
            logger.log(f"  Warning: Missing {ratings_file}")
            continue

        df = pd.read_csv(ratings_file, sep="\t")
        df["category"] = cat
        df["screenshot_path"] = df["stimulusId"].apply(lambda s: os.path.join(screenshot_dir, s))
        df["rating"] = df["AE"]
        df = df[df["screenshot_path"].apply(os.path.exists)]
        rows.append(df[["stimulusId", "category", "screenshot_path", "rating"]])

    df_all = pd.concat(rows, ignore_index=True)
    df_all = df_all.drop_duplicates("screenshot_path").reset_index(drop=True)

    logger.log(f"  Total: {len(df_all)} samples")
    logger.log(f"  Rating range: [{df_all['rating'].min():.2f}, {df_all['rating'].max():.2f}]")
    logger.log(f"  Category distribution:")
    for cat in CFG.CATEGORIES:
        n = (df_all["category"] == cat).sum()
        logger.log(f"    {cat}: {n}")

    return df_all

# ============================================================
# Dataset
# ============================================================
def crop_and_resize(img, size, above_fold_height=None):
    """
    Crop the screenshot to its top `above_fold_height` pixels (the
    above-the-fold region), then letterbox to a square and resize to
    `size` x `size`. Pass above_fold_height=None to keep the full page.
    """
    w, h = img.size

    # Step 1: Crop to above-the-fold if specified
    if above_fold_height is not None and h > above_fold_height:
        img = img.crop((0, 0, w, above_fold_height))
        h = above_fold_height

    # Step 2: Letterbox to square
    side = max(w, h)
    canvas = Image.new("RGB", (side, side), (128, 128, 128))
    canvas.paste(img, ((side - w) // 2, (side - h) // 2))

    # Step 3: Resize to target size
    return canvas.resize((size, size), Image.BICUBIC)


def get_cache_path(src_path, category):
    basename = os.path.splitext(os.path.basename(src_path))[0]
    return os.path.join(CFG.CACHE_DIR, f"wdp_{category}_{basename}_{CFG.IMG_SIZE}.jpg")


class WDPDataset(Dataset):
    def __init__(self, df, mean, std):
        self.df = df.reset_index(drop=True)
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cache_path = get_cache_path(row["screenshot_path"], row["category"])

        if os.path.exists(cache_path):
            img = Image.open(cache_path).convert("RGB")
        else:
            img = Image.open(row["screenshot_path"]).convert("RGB")
            img = crop_and_resize(img, CFG.IMG_SIZE, CFG.ABOVE_FOLD_HEIGHT)
            img.save(cache_path, "JPEG", quality=95)
            img = Image.open(cache_path).convert("RGB")

        x = TF.to_tensor(img)
        x = (x - torch.tensor(self.mean).view(-1,1,1)) / torch.tensor(self.std).view(-1,1,1)
        y = torch.tensor([row["rating_norm"]], dtype=torch.float32)

        return {"pixel_values": x, "labels": y, "idx": idx}

# ============================================================
# Metrics
# ============================================================
def compute_metrics(y_true, y_pred):
    return {
        "r": float(pearsonr(y_pred, y_true)[0]),
        "rho": float(spearmanr(y_pred, y_true)[0]),
        "rmse": float(np.sqrt(np.mean((y_pred - y_true)**2))),
    }


def bootstrap_ci(y_true, y_pred, metric_fn, n=2000, alpha=0.05):
    rng = np.random.default_rng(42)
    vals = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        vals.append(metric_fn(y_true[idx], y_pred[idx]))
    return np.percentile(vals, [100*alpha/2, 100*(1-alpha/2)])

# ============================================================
# Model
# ============================================================
def load_fresh_model():
    model = AutoModelForImageClassification.from_pretrained(
        CFG.HF_MODEL, num_labels=1, problem_type="regression", ignore_mismatched_sizes=True
    )
    model.to(device)

    try:
        proc = AutoImageProcessor.from_pretrained(CFG.HF_MODEL)
        mean, std = proc.image_mean, proc.image_std
    except:
        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

    return model, mean, std


@torch.no_grad()
def predict(model, loader, use_tta=True):
    model.eval()
    preds, indices = [], []
    with amp.autocast('cuda', enabled=use_cuda):
        for batch in loader:
            x = batch["pixel_values"].to(device)
            if use_tta:
                out = (model(x).logits + model(torch.flip(x, [-1])).logits) / 2
            else:
                out = model(x).logits
            preds.append(out.squeeze(1).cpu().numpy())
            indices.extend(batch["idx"].numpy().tolist())
    return np.concatenate(preds), indices

# ============================================================
# Training
# ============================================================
def loss_function(pred, target):
    return F.mse_loss(pred, target)


def train_fold(model, train_loader, val_loader, fold_num):
    """Train one fold with 3-phase training."""

    def freeze_all(m, freeze=True):
        for p in m.parameters():
            p.requires_grad = not freeze

    def get_head_params(m):
        return [n for n, _ in m.named_parameters()
                if any(n.startswith(p) for p in ["classifier", "head", "fc", "score"])]

    def unfreeze_head(m):
        freeze_all(m, True)
        for n, p in m.named_parameters():
            if n in get_head_params(m):
                p.requires_grad = True

    def unfreeze_last_k(m, k=4):
        freeze_all(m, True)
        for n, p in m.named_parameters():
            if n in get_head_params(m):
                p.requires_grad = True
        if hasattr(m, "vision_model") and hasattr(m.vision_model, "encoder"):
            blocks = list(m.vision_model.encoder.layers)
            for i in range(max(0, len(blocks)-k), len(blocks)):
                for p in blocks[i].parameters():
                    p.requires_grad = True

    def unfreeze_all(m):
        freeze_all(m, False)

    best_r = -float("inf")
    best_state = None

    def run_phase(name, epochs, lr, unfreeze_fn):
        nonlocal best_r, best_state

        unfreeze_fn(model)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                               lr=lr, weight_decay=CFG.WEIGHT_DECAY)
        steps = epochs * len(train_loader)
        sch = get_cosine_schedule_with_warmup(opt, int(CFG.WARMUP_RATIO * steps), steps)
        scaler = amp.GradScaler(enabled=use_cuda)
        patience = 0

        for epoch in range(1, epochs + 1):
            model.train()
            for batch in train_loader:
                x = batch["pixel_values"].to(device)
                y = batch["labels"].to(device)

                with amp.autocast('cuda', enabled=use_cuda):
                    loss = loss_function(model(x).logits, y)

                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad()
                sch.step()

            # Validate
            y_pred, _ = predict(model, val_loader, use_tta=False)
            y_true = val_loader.dataset.df["rating_norm"].values
            r = pearsonr(y_pred, y_true)[0]

            if r > best_r:
                best_r = r
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= 5:
                    break

        if best_state:
            model.load_state_dict(best_state)

    # 3-phase training
    run_phase("P1", CFG.EPOCHS_HEAD, CFG.HEAD_LR, unfreeze_head)
    run_phase("P2", CFG.EPOCHS_PARTIAL, CFG.BASE_LR, lambda m: unfreeze_last_k(m, 4))
    run_phase("P3", CFG.EPOCHS_FULL, CFG.BASE_LR, unfreeze_all)

    return model

# ============================================================
# Main: Stratified 5-Fold CV
# ============================================================
def main():
    logger.log("="*60)
    logger.log("STRATIFIED 5-FOLD CV ON WDP DATASET WITH ABOVE FOLD CROPPED IMAGES")
    logger.log("~3136 samples across 4 categories")
    logger.log("="*60)

    # Load data
    df = load_wdp_data()

    # Precache all images
    logger.log("\nPrecaching images...")
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Caching", leave=True):
        cache_path = get_cache_path(row["screenshot_path"], row["category"])
        if not os.path.exists(cache_path):
            img = Image.open(row["screenshot_path"]).convert("RGB")
            img = crop_and_resize(img, CFG.IMG_SIZE, CFG.ABOVE_FOLD_HEIGHT)
            img.save(cache_path, "JPEG", quality=95)
    logger.log("  Caching complete!")

    # Get normalization stats
    _, mean, std = load_fresh_model()

    # Z-score the ratings; Pearson r is invariant to this affine rescaling
    mu, sd = df["rating"].mean(), df["rating"].std()
    df["rating_norm"] = (df["rating"] - mu) / sd
    logger.log(f"\nRating normalization: mean={mu:.3f}, std={sd:.3f}")

    # Stratified 5-Fold CV (stratify by category)
    skfold = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    stratify_labels = df["category"].values

    fold_results = []
    category_results = {cat: [] for cat in CFG.CATEGORIES}
    all_predictions = np.zeros(len(df))
    all_true = df["rating_norm"].values

    for fold, (train_idx, test_idx) in enumerate(skfold.split(df, stratify_labels), 1):
        logger.log(f"\n{'='*60}")
        logger.log(f"FOLD {fold}/{CFG.N_FOLDS}")
        logger.log(f"{'='*60}")

        # Check stratification
        test_cats = df.iloc[test_idx]["category"].value_counts()
        logger.log(f"Train: {len(train_idx)}, Test: {len(test_idx)}")
        logger.log(f"Test distribution: {dict(test_cats)}")

        # Split train into train/val (stratified)
        train_df_temp = df.iloc[train_idx]

        # Validation split: first fold of a 6-way stratified split of the training portion
        val_skf = StratifiedKFold(n_splits=int(1/0.15), shuffle=True, random_state=CFG.SEED + fold)
        for train_idx_inner, val_idx_inner in val_skf.split(train_df_temp, train_df_temp["category"]):
            break  # Just take first split

        train_idx_final = train_idx[train_idx_inner]
        val_idx_final = train_idx[val_idx_inner]

        df_train = df.iloc[train_idx_final].reset_index(drop=True)
        df_val = df.iloc[val_idx_final].reset_index(drop=True)
        df_test = df.iloc[test_idx].reset_index(drop=True)

        logger.log(f"  Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

        # Load fresh model
        model, mean, std = load_fresh_model()

        # Create datasets
        train_ds = WDPDataset(df_train, mean, std)
        val_ds = WDPDataset(df_val, mean, std)
        test_ds = WDPDataset(df_test, mean, std)

        train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                                  num_workers=0, pin_memory=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE*2, shuffle=False,
                               num_workers=0, pin_memory=True)
        test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE*2, shuffle=False,
                                num_workers=0, pin_memory=True)

        # Train
        logger.log(f"\nTraining fold {fold}...")
        model = train_fold(model, train_loader, val_loader, fold)

        # Test
        y_pred, pred_indices = predict(model, test_loader, use_tta=CFG.USE_TTA)
        y_true = df_test["rating_norm"].values

        # Store predictions
        for i, pred_idx in enumerate(pred_indices):
            all_predictions[test_idx[pred_idx]] = y_pred[i]

        # Overall metrics
        metrics = compute_metrics(y_true, y_pred)

        fold_results.append({
            "fold": fold,
            "r": metrics["r"],
            "rho": metrics["rho"],
            "rmse": metrics["rmse"],
            "n_test": len(test_idx)
        })

        logger.log(f"\nFold {fold} Overall Results:")
        logger.log(f"  r      = {metrics['r']:.4f}")
        logger.log(f"  ρ      = {metrics['rho']:.4f}")
        logger.log(f"  RMSE   = {metrics['rmse']:.4f}")

        # Per-category metrics
        logger.log(f"\nFold {fold} Per-Category:")
        for cat in CFG.CATEGORIES:
            mask = df_test["category"] == cat
            if mask.sum() > 0:
                cat_true = y_true[mask]
                cat_pred = y_pred[mask]
                cat_r = pearsonr(cat_pred, cat_true)[0]
                cat_rho = spearmanr(cat_pred, cat_true)[0]
                category_results[cat].append(cat_r)
                logger.log(f"  {cat:>12}: r={cat_r:.3f}, ρ={cat_rho:.3f} (n={mask.sum()})")

        # Cleanup
        del model, train_ds, val_ds, test_ds
        del train_loader, val_loader, test_loader
        torch.cuda.empty_cache()
        gc.collect()

    # ============================================================
    # Aggregate Results
    # ============================================================
    logger.log("\n" + "="*60)
    logger.log("STRATIFIED 5-FOLD CV RESULTS - WDP")
    logger.log("="*60)

    r_scores = [f["r"] for f in fold_results]
    rho_scores = [f["rho"] for f in fold_results]

    logger.log(f"\nPer-fold r:   {[f'{x:.3f}' for x in r_scores]}")
    logger.log(f"Per-fold ρ:   {[f'{x:.3f}' for x in rho_scores]}")

    mean_r = np.mean(r_scores)
    std_r = np.std(r_scores)
    mean_rho = np.mean(rho_scores)
    std_rho = np.std(rho_scores)

    logger.log(f"\nAggregated (mean ± std):")
    logger.log(f"  r   = {mean_r:.4f} ± {std_r:.4f}")
    logger.log(f"  ρ   = {mean_rho:.4f} ± {std_rho:.4f}")

    # Overall metrics from all predictions
    overall_r = pearsonr(all_predictions, all_true)[0]
    overall_rho = spearmanr(all_predictions, all_true)[0]

    logger.log(f"\nOverall (all predictions combined):")
    logger.log(f"  r   = {overall_r:.4f}")
    logger.log(f"  ρ   = {overall_rho:.4f}")

    # Bootstrap CI on overall predictions
    r_ci = bootstrap_ci(all_true, all_predictions, lambda t, p: pearsonr(p, t)[0])
    logger.log(f"\nBootstrap 95% CI (overall):")
    logger.log(f"  r:   [{r_ci[0]:.3f}, {r_ci[1]:.3f}]")

    # Per-category aggregated
    logger.log(f"\nPer-Category Results (mean ± std across folds):")
    category_summary = {}
    for cat in CFG.CATEGORIES:
        cat_r_mean = np.mean(category_results[cat])
        cat_r_std = np.std(category_results[cat])
        category_summary[cat] = {"mean_r": cat_r_mean, "std_r": cat_r_std}
        logger.log(f"  {cat:>12}: r = {cat_r_mean:.3f} ± {cat_r_std:.3f}")

    # Save results
    results = {
        "dataset": "WDP",
        "n_samples": len(df),
        "n_folds": CFG.N_FOLDS,
        "stratified": True,
        "fold_results": fold_results,
        "mean_r": mean_r,
        "std_r": std_r,
        "mean_rho": mean_rho,
        "std_rho": std_rho,
        "overall_r": overall_r,
        "overall_rho": overall_rho,
        "r_ci": r_ci.tolist(),
        "category_results": category_summary,
        "config": {
            "epochs": f"{CFG.EPOCHS_HEAD}/{CFG.EPOCHS_PARTIAL}/{CFG.EPOCHS_FULL}",
            "base_lr": CFG.BASE_LR,
            "head_lr": CFG.HEAD_LR,
        }
    }

    results_path = os.path.join(CFG.RESULTS_DIR, "wdp_5fold_results.json")
    with open(results_path, "w") as f:
        json.dump(results, f, indent=2)
    logger.log(f"\nSaved: {results_path}")

    # Save predictions
    pred_df = df[["stimulusId", "category", "rating"]].copy()
    pred_df["prediction_norm"] = all_predictions
    pred_df["prediction"] = all_predictions * sd + mu
    pred_path = os.path.join(CFG.RESULTS_DIR, "wdp_5fold_predictions.csv")
    pred_df.to_csv(pred_path, index=False)
    logger.log(f"Saved predictions: {pred_path}")

    logger.log("\n" + "="*60)
    logger.log("DONE")
    logger.log("="*60)

    return results


if __name__ == "__main__":
    main()

Device: cuda
Mounted at /content/drive
[21:17:22] ============================================================
[21:17:22] STRATIFIED 5-FOLD CV ON WDP DATASET WITH ABOVE FOLD CROPPED IMAGES
[21:17:22] ~3136 samples across 4 categories
[21:17:22] ============================================================
[21:17:22] Loading WDP dataset...
[21:17:23]   Total: 3136 samples
[21:17:23]   Rating range: [-2.95, 1.99]
[21:17:23]   Category distribution:
[21:17:23]     banks: 1028
[21:17:23]     fashion: 547
[21:17:23]     homeware: 505
[21:17:23]     universities: 1056
[21:17:23] 
Precaching images...


Caching:   0%|          | 0/3136 [00:00<?, ?it/s]

[21:21:12]   Caching complete!


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

...

[21:30:01] 
Training fold 3...
[21:34:16] 
Fold 3 Overall Results:
[21:34:16]   r      = 0.6047
[21:34:16]   ρ      = 0.5557
[21:34:16]   RMSE   = 0.7967
[21:34:16] 
Fold 3 Per-Category:
[21:34:16]          banks: r=0.556, ρ=0.496 (n=206)
[21:34:16]        fashion: r=0.413, ρ=0.418 (n=109)
[21:34:16]       homeware: r=0.498, ρ=0.400 (n=101)
[21:34:16]   universities: r=0.748, ρ=0.704 (n=211)
[21:34:16] 
[21:34:16] FOLD 4/5
[21:34:16] ============================================================
[21:34:16] Train: 2509, Test: 627
[21:34:16] Test distribution: {'universities': np.int64(211), 'banks': np.int64(205), 'fashion': np.int64(110), 'homeware': np.int64(101)}
[21:34:16]   Train: 2090, Val: 419, Test: 627


Some weights of CLIPForImageClassification were not initialized from the model checkpoint at openai/clip-vit-base-patch16 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[21:34:17] 
Training fold 4...
[21:38:31] 
Fold 4 Overall Results:
[21:38:31]   r      = 0.5947
[21:38:31]   ρ      = 0.5528
[21:38:31]   RMSE   = 0.7888
[21:38:31] 
Fold 4 Per-Category:
[21:38:31]          banks: r=0.596, ρ=0.574 (n=205)
[21:38:31]        fashion: r=0.328, ρ=0.294 (n=110)
[21:38:31]       homeware: r=0.516, ρ=0.501 (n=101)
[21:38:31]   universities: r=0.683, ρ=0.654 (n=211)
[21:38:32] 
[21:38:32] FOLD 5/5
[21:38:32] ============================================================
[21:38:32] Train: 2509, Test: 627
[21:38:32] Test distribution: {'universities': np.int64(211), 'banks': np.int64(205), 'fashion': np.int64(110), 'homeware': np.int64(101)}
[21:38:32]   Train: 2090, Val: 419, Test: 627


Some weights of CLIPForImageClassification were not initialized from the model checkpoint at openai/clip-vit-base-patch16 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[21:38:33] 
Training fold 5...
[21:42:46] 
Fold 5 Overall Results:
[21:42:47]   r      = 0.6417
[21:42:47]   ρ      = 0.5906
[21:42:47]   RMSE   = 0.7720
[21:42:47] 
Fold 5 Per-Category:
[21:42:47]          banks: r=0.654, ρ=0.547 (n=205)
[21:42:47]        fashion: r=0.487, ρ=0.466 (n=110)
[21:42:47]       homeware: r=0.534, ρ=0.486 (n=101)
[21:42:47]   universities: r=0.725, ρ=0.711 (n=211)
[21:42:47] 
[21:42:47] STRATIFIED 5-FOLD CV RESULTS - WDP
[21:42:47] ============================================================
[21:42:47] 
Per-fold r:   ['0.581', '0.612', '0.605', '0.595', '0.642']
[21:42:47] Per-fold ρ:   ['0.552', '0.555', '0.556', '0.553', '0.591']
[21:42:47] 
Aggregated (mean ± std):
[21:42:47]   r   = 0.6070 ± 0.0202
[21:42:47]   ρ   = 0.5612 ± 0.0148
[21:42:47] 
Overall (all predictions combined):
[21:42:47]   r   = 0.6062
[21:42:47]   ρ   = 0.5632
[21:42:48] 
Bootstrap 95% CI (overall):
[21:42:48]   r:   [0.582, 0.630]
[21:42:48] 
Per-Category Results (mean ± std across 